# Assignment: Pricing Agent with RAG — Chunking Strategies

## Learning Objectives
In this assignment you will build a **pricing agent** using Retrieval-Augmented Generation (RAG).
By the end, you will be able to:
- Apply four different chunking strategies to a pricing document
- Compare how chunk size affects retrieval quality
- Build a full RAG pipeline using the free **Groq API** (LLaMA 3.1) and **HuggingFace embeddings**
- Implement HyDE to improve retrieval for vague pricing queries

## Setup
- Get your **free Groq API key** at https://console.groq.com
- Complete each `# TODO` section in the code cells below

## Grading
| Task | Points |
|---|---|
| Task 1: Fixed-Length Chunking | 10 |
| Task 2: Recursive Text Splitting | 20 |
| Task 3: Markdown Header Chunking | 20 |
| Task 4: Build Vector Stores | 20 |
| Task 5: RAG Chain with Groq | 20 |
| Bonus: Semantic Chunking + HyDE | 10 |

> **Tip:** Run each cell in order. Read the markdown explanations before attempting the code.

In [1]:
# Install all required packages
# These are already provided for you — just run this cell
!pip install -q \
    langchain \
    langchain-community \
    langchain-groq \
    langchain-experimental \
    langchain-chroma \
    langchain-text-splitters \
    langchain-huggingface \
    sentence-transformers \
    chromadb \
    rank_bm25 \
    numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/2

In [ ]:
import os
import getpass

# TODO: Set up your Groq API key
# Get a free key at https://console.groq.com
# This code checks if the key is already set; if not, prompts you to enter it

# YOUR CODE HERE
# Hint: Use the same pattern as the original notebook but with GROQ_API_KEY

print("API key configured.")

## The Pricing Document

The pricing document below is already written for you — this is your **knowledge base**.
Read through it to understand its structure before implementing the chunking strategies.

Notice:
- It uses markdown headers (`#`, `##`, `###`) to organize products and plans
- Each plan has a clear price and a bullet list of features
- Some sections (billing, compliance) describe policies rather than plan features

**Question to think about:** Which chunking strategy do you think will work best for this document structure and why?

In [ ]:
# Pricing document — already provided, just run this cell
pricing_content = """
# DataFlow Platform - Pricing Guide

## DataFlow Analytics

### Starter Plan - $29/month
- Up to 5 users
- 10GB data storage
- Basic dashboards (up to 10)
- CSV and Excel data import
- Email support with 48-hour response time
- Data retention: 90 days
- API access: Not included
- Custom branding: Not included

### Pro Plan - $99/month
- Up to 25 users
- 100GB data storage
- Unlimited dashboards
- All data source integrations (50+ connectors)
- Priority email and chat support with 4-hour response time
- Data retention: 1 year
- API access: 10,000 calls/month
- Custom branding: Included
- Advanced analytics: Predictive modeling and anomaly detection
- Scheduled reports: Up to 50 per month

### Enterprise Plan - Custom Pricing (starting at $500/month)
- Unlimited users
- 10TB or more data storage
- Unlimited dashboards
- All integrations plus custom connectors
- 24/7 dedicated support with SLA guarantee
- Data retention: 7 years
- API access: Unlimited
- Custom branding: Full white-label
- SSO and SAML authentication
- On-premise deployment option
- Dedicated account manager

## DataFlow CRM

### Free Plan - $0/month
- Up to 2 users
- 1,000 contacts
- Basic pipeline management
- Email integration with Gmail and Outlook
- Community support only

### Growth Plan - $49/user/month
- Unlimited users
- 50,000 contacts
- Advanced pipeline management
- Email automation: Up to 5 sequences
- Standard reports
- 20+ app integrations including Slack and Zapier
- Chat and email support during business hours

### Scale Plan - $99/user/month
- Unlimited users and contacts
- Custom pipeline stages
- Full email and SMS integration
- Advanced automation builder
- AI-powered lead scoring
- Custom reports and dashboards
- 100+ app integrations
- Priority 24/7 support

### Enterprise Plan - $199/user/month
- Everything in Scale plan
- Dedicated customer success manager
- Custom AI models trained on your business data
- Advanced security including SOC2 and HIPAA compliance
- Custom SLA and dedicated infrastructure
- API access: Unlimited
- Revenue intelligence and forecasting

## DataFlow Storage

### Basic Plan - $9/month
- 500GB storage
- 1TB monthly transfer
- Basic CDN with 5 edge locations
- 99.9% uptime SLA
- Standard AES-256 encryption
- Web-based management console

### Standard Plan - $29/month
- 5TB storage
- 10TB monthly transfer
- Advanced CDN with 50+ edge locations
- 99.95% uptime SLA
- API access and SDK
- Version control with 30-day history
- Automated daily backups

### Premium Plan - $79/month
- 25TB storage
- Unlimited transfer
- Premium CDN with 150+ edge locations
- 99.99% uptime SLA
- Enterprise encryption with HSM support
- Version control with 1-year history
- Automated backups with cross-region replication
- Compliance reports for SOC2 and ISO 27001

## Bundles and Add-ons

### DataFlow Suite Bundle - $149/month
Combines Analytics Pro, CRM Growth, and Storage Standard into one package.
Save 35% compared to purchasing each product individually.
Best for growing teams that need complete data infrastructure.

### Security Add-on - $19/month
Advanced threat detection and SIEM integration.
Custom security policies and audit logging.
Available as an add-on for all DataFlow products.

### Training and Onboarding Package - $499 one-time fee
10 hours of guided onboarding sessions.
Custom workflow setup for your team.
Team training sessions included.
30 days of dedicated post-onboarding support.

## Billing and Policies

### Payment Options
Monthly and annual billing are both available.
Annual plans offer a 20% discount compared to monthly billing.
Accepted payment methods include major credit cards, PayPal, and wire transfers for Enterprise plans.

### Cancellation Policy
Monthly plans can be cancelled at any time and take effect immediately.
Annual plans require 30-day notice and are non-refundable after the 14-day grace period.
Upgrades take effect immediately with prorated credit for unused time.
Downgrades take effect at the end of the current billing cycle.

### Free Trials
All paid plans include a 14-day free trial with no credit card required.
Enterprise plans include a 30-day proof-of-concept period with dedicated support.
Trial accounts have full access to all features of the selected plan.

### Compliance and Security
All DataFlow products are SOC2 Type II certified.
HIPAA compliance is available on Enterprise plans only.
GDPR compliance tools are included in all plans at no extra cost.
All data is stored in ISO 27001 certified data centers.
"""

with open("pricing_guide.md", "w") as f:
    f.write(pricing_content)

print(f"Pricing guide saved: {len(pricing_content)} characters")
print("Read through the document above before continuing!")

## Setting Up Free Embeddings

We use **HuggingFace sentence-transformers** instead of OpenAI embeddings because:
- They run **locally** — no API calls or API key needed
- They are completely **free**
- `all-MiniLM-L6-v2` is fast and accurate for semantic similarity

**Your task:** Initialize the embeddings model and verify it works by embedding a test sentence.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# TODO: Initialize HuggingFace embeddings
# Use model_name="all-MiniLM-L6-v2" and model_kwargs={"device": "cpu"}

embeddings = None  # Replace with your implementation

# TODO: Verify the embeddings work by embedding a test query
# Embed the string "What is the price of the Pro plan?"
# Print the dimension of the resulting vector

# YOUR CODE HERE

## Task 1: Fixed-Length Chunking (10 pts)

Fixed-length chunking splits text at a fixed character count, regardless of content structure.
It is the simplest strategy but can break pricing plan descriptions mid-sentence or mid-feature-list.

**Instructions:**
1. Use `CharacterTextSplitter` with `chunk_size=300`, `chunk_overlap=30`, and `separator="\n"`
2. Split `pricing_content`
3. Print the total number of chunks produced
4. Print the first 3 chunks with their lengths

**Reflection question (answer in the markdown cell below):** Did any chunks get cut in the middle of a pricing plan? How can you tell?

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

# TODO: Task 1 - Implement fixed-length chunking
#
# Step 1: Create a CharacterTextSplitter with:
#   - chunk_size = 300
#   - chunk_overlap = 30
#   - separator = "\n"
#
# Step 2: Split pricing_content using .split_text()
#
# Step 3: Print the total number of chunks
#
# Step 4: Print the first 3 chunks with their character lengths

# YOUR CODE HERE

**Your reflection (Task 1):**

_Write your answer here: Did any chunks get cut mid-plan? How did fixed-length chunking handle the pricing data?_

## Task 2: Recursive Character Text Splitting (20 pts)

Recursive splitting tries natural separators in order: `\n\n` → `\n` → ` `.
It is more intelligent than fixed-length because it respects paragraph boundaries.

**Instructions:**
1. Load `pricing_guide.md` using `TextLoader`
2. Split with **three different chunk sizes**: `[100, 300, 600]`, using 10% overlap for each
3. Store results in a dict `chunked_documents = {size: chunks, ...}`
4. For each size, print: number of chunks and average chunk length
5. For chunk size 300, print the chunk(s) that contain `"Pro Plan"` — does the full plan fit in one chunk?

**Reflection question:** For a pricing agent, what chunk size would you expect works best and why?

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO: Task 2 - Implement recursive character text splitting
#
# Step 1: Load pricing_guide.md with TextLoader
#   documents = ...
#
# Step 2: For each size in [100, 300, 600]:
#   - Create a RecursiveCharacterTextSplitter with chunk_size=size, chunk_overlap=int(size*0.1)
#   - Split the documents
#   - Store in chunked_documents[size]
#   - Print: size, number of chunks, average chunk length
#
# Step 3: For chunk size 300, find and print chunks that contain "Pro Plan"

chunk_sizes = [100, 300, 600]
chunked_documents = {}

# YOUR CODE HERE

**Your reflection (Task 2):**

_Which chunk size kept the Pro Plan description intact? Which size split it across multiple chunks? What is the trade-off?_

## Task 3: Markdown Header Chunking (20 pts)

Markdown header splitting splits on `#`, `##`, `###` markers and preserves the header hierarchy as **metadata**.

This is particularly powerful for pricing documents because:
- Each plan (`###`) becomes one chunk
- The product name (`##`) is stored in chunk metadata
- You can later filter by product: `metadata['Product'] == 'DataFlow CRM'`

**Instructions:**
1. Use `MarkdownHeaderTextSplitter` with three header levels:
   - `#` → `"Platform"`
   - `##` → `"Product"`
   - `###` → `"Plan"`
2. Split `pricing_content`
3. Print all chunks with their metadata
4. Count how many chunks belong to each `Product` in the metadata

**Reflection question:** Why is the metadata hierarchy valuable for a pricing agent?

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# TODO: Task 3 - Implement markdown header chunking
#
# Step 1: Define headers_to_split_on as a list of tuples:
#   [("#", "Platform"), ("##", "Product"), ("###", "Plan")]
#
# Step 2: Create a MarkdownHeaderTextSplitter with those headers
#
# Step 3: Split pricing_content
#
# Step 4: Print each chunk with its metadata
#
# Step 5: Count and print how many chunks belong to each Product
#   Hint: use doc.metadata.get("Product", "No Product") and a Counter or dict

# YOUR CODE HERE

**Your reflection (Task 3):**

_How does the metadata help? Could you use it to build a product-specific pricing agent? How?_

## Task 4: Build Vector Stores & Compare Retrieval (20 pts)

Now we'll store the recursively-split chunks in ChromaDB vector stores and compare how different chunk sizes affect retrieval.

**Instructions:**
1. For each chunk size in `chunked_documents`, create a ChromaDB vector store:
   - Use the `embeddings` object from earlier
   - Persist to `./pricing_chroma_{size}` (e.g., `./pricing_chroma_300`)
   - Store each in `vector_dbs[size]`
2. Run the query `"What features are included in the Analytics Pro plan?"` against all three vector stores (k=3)
3. Print the top-3 results for each chunk size
4. Determine which chunk size returns the most complete answer

In [ ]:
from langchain_chroma import Chroma
import shutil

# TODO: Task 4 - Build ChromaDB vector stores and compare retrieval
#
# Step 1: Clean up any existing DB directories from previous runs
#   Hint: os.path.exists() and shutil.rmtree()
#
# Step 2: For each size in chunked_documents:
#   - Create a Chroma vector store using Chroma.from_documents()
#   - Parameters: documents=chunks, embedding=embeddings, persist_directory=f"./pricing_chroma_{size}"
#   - Store in vector_dbs[size]
#   - Print a confirmation with the number of indexed documents
#
# Step 3: Define query = "What features are included in the Analytics Pro plan?"
#
# Step 4: For each vector_db, run similarity_search(query, k=3)
#   Print the results for each chunk size

vector_dbs = {}

# YOUR CODE HERE

**Your reflection (Task 4):**

_Which chunk size gave the best results for the pricing query? Did small chunks miss important context? Did large chunks return irrelevant content?_

## Task 5: Build the Pricing Agent RAG Chain (20 pts)

Combine everything into a working pricing agent using **Groq's free LLaMA 3.1 API**.

**Instructions:**
1. Initialize `ChatGroq` with model `"llama-3.1-8b-instant"`, `temperature=0`, `max_tokens=1024`
2. Create a retriever from `vector_dbs[300]` with `k=4`
3. Write a system prompt that:
   - Tells the LLM it is a pricing assistant for DataFlow Platform
   - Instructs it to use ONLY the provided context
   - Asks it to be precise with prices
   - Has a placeholder `{context}` and `{question}`
4. Build the RAG chain using LangChain Expression Language (LCEL):
   `{context: retriever | format_docs, question: passthrough} | prompt | llm | parser`
5. Test with at least 3 pricing questions of your choice

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# TODO: Task 5a - Initialize the Groq LLM
# Use model="llama-3.1-8b-instant", temperature=0, max_tokens=1024

llm = None  # Replace with your implementation

# TODO: Task 5b - Create a retriever from vector_dbs[300] with k=4

retriever = None  # Replace with your implementation

# TODO: Task 5c - Write the pricing agent system prompt
# It should instruct the LLM to:
#   - Act as a pricing assistant for DataFlow Platform
#   - Answer using ONLY the provided context
#   - Be precise with prices and plan names
#   - Say "I don't have that information" if context doesn't cover the question
# Include {context} and {question} placeholders

pricing_prompt = None  # Replace with your ChatPromptTemplate

# Helper function to format retrieved documents into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# TODO: Task 5d - Build the RAG chain using LCEL
# Chain: {context: retriever|format_docs, question: passthrough} | prompt | llm | StrOutputParser()

rag_chain = None  # Replace with your implementation

print("RAG chain built!" if rag_chain else "RAG chain not yet implemented.")

In [ ]:
# TODO: Task 5e - Test your pricing agent
#
# Write at least 3 pricing questions and invoke rag_chain with each.
# Suggested questions (feel free to use your own):
#   - "What is included in the Analytics Pro plan?"
#   - "Which plan should I choose for a team of 10 that needs CRM?"
#   - "What is the annual discount and how does it apply?"
#   - "Which plans include HIPAA compliance?"

my_questions = [
    # Add your questions here as strings
]

# YOUR CODE HERE — invoke rag_chain for each question and print Q + A

**Your reflection (Task 5):**

_Did the agent answer correctly? Were there any questions where it struggled or hallucinated? What could you improve?_

## Bonus: Semantic Chunking + HyDE (10 pts)

### Part A: Semantic Chunking
Semantic chunking uses embeddings to find natural topic boundaries. Unlike fixed or recursive splitting, it groups sentences with similar meaning together.

**Instructions:**
- Use `SemanticChunker` from `langchain_experimental.text_splitter`
- Pass your HuggingFace `embeddings` object
- Use `breakpoint_threshold_type="percentile"` and `breakpoint_threshold_amount=70`
- Print all resulting chunks with their lengths

### Part B: HyDE (Hypothetical Document Embeddings)
HyDE improves BM25 retrieval for vague queries by asking the LLM to generate a hypothetical answer first, then searching with that richer text.

**Instructions:**
1. Build a BM25 index over chunks split at size 250
2. Test with `query = "affordable option for a small startup with basic needs"`
3. Show retrieval results WITHOUT HyDE
4. Use Groq LLM to generate a hypothetical pricing answer for the query
5. Combine query + hypothetical answer and search again
6. Compare and explain the difference

In [ ]:
# BONUS Part A: Semantic Chunking
from langchain_experimental.text_splitter import SemanticChunker

# TODO: Initialize SemanticChunker with HuggingFace embeddings
# Use breakpoint_threshold_type="percentile" and breakpoint_threshold_amount=70

# TODO: Create semantic chunks from pricing_content

# TODO: Print the number of chunks and the content/length of each chunk

# YOUR CODE HERE

In [ ]:
# BONUS Part B: HyDE (Hypothetical Document Embeddings)
import numpy as np
from rank_bm25 import BM25Okapi

vague_query = "affordable option for a small startup with basic needs"

# TODO: Step 1 - Build BM25 index
# - Use RecursiveCharacterTextSplitter with chunk_size=250, chunk_overlap=25
# - Split the loaded documents
# - Extract text from each chunk
# - Tokenize (lowercase + split) and build BM25Okapi

# YOUR CODE HERE

# TODO: Step 2 - Retrieve WITHOUT HyDE
# - Tokenize vague_query
# - Get BM25 scores and find top 3 results
# - Print them with their scores

# YOUR CODE HERE

# TODO: Step 3 - Generate hypothetical answer using Groq
# - Create a prompt asking the LLM to generate a realistic pricing plan description
#   that would answer the vague_query
# - Invoke the llm and print the hypothetical answer

# YOUR CODE HERE

# TODO: Step 4 - Retrieve WITH HyDE
# - Combine vague_query + hypothetical_answer into one string
# - Get BM25 scores and print top 3 results with scores

# YOUR CODE HERE

# TODO: Step 5 - Print a comparison / interpretation

**Your reflection (Bonus):**

_Semantic chunking: How did it group the pricing content? Were related policies chunked together?_

_HyDE: How did the hypothetical answer change the retrieval results? What vocabulary did it add?_

## Submission Checklist

Before submitting, verify each item:

- [ ] All cells run without errors (Kernel → Restart & Run All)
- [ ] Task 1: Fixed-length chunks printed with lengths
- [ ] Task 2: Three chunk sizes compared with statistics
- [ ] Task 3: Markdown chunks printed with metadata; product count included
- [ ] Task 4: Vector stores built; retrieval comparison printed
- [ ] Task 5: RAG chain built; at least 3 pricing questions answered
- [ ] All reflection cells filled in with your observations
- [ ] (Optional) Bonus: Semantic chunking + HyDE implemented and compared

**Final question:** Based on all four strategies you implemented, which chunking approach would you recommend for a production pricing agent and why? Write 2-3 sentences below.

_Your answer:_